# 🏗️ Data Engineering — A Clear Introduction (with examples)

A beginner-friendly tour of **modern data engineering**, from the tiny
transaction at a coffee counter to the executive dashboard. Every concept comes
with a plain-English explanation, a worked **example**, and — where it helps — a
small **runnable PySpark cell** you can execute on Databricks.

We follow one running story: **BrewBox**, a coffee-shop chain with 500 stores.
Every second something happens — a latte is sold, a loyalty point is added, a
store runs low on milk — and all of it creates data.

**How to use it**
1. Import into Databricks (*Workspace → Import → File*) and attach compute.
2. Read top to bottom — each topic builds on the previous one.
3. Run the code cells to see the ideas in action.

**Covered:** OLTP & OLAP · Data Warehouse · Data Lake · Lakehouse · dimensional
modeling (**fact & dimension**, **star** & **snowflake** schemas) · ETL & ELT ·
Apache Spark · data pipelines · the **Medallion Architecture** (Bronze → Silver
→ Gold) · and the **technology landscape** (SQL, Python, pandas, PySpark, dbt,
Airflow, and more).

In [ ]:
# Databricks pre-injects `spark`. This fallback lets the notebook also run
# anywhere PySpark is installed. Run me first.
try:
    spark
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("de-intro").getOrCreate()

from pyspark.sql import functions as F, Window
print("Spark:", spark.version)

## 1 · What is data engineering?

On its own, raw data is messy and scattered across dozens of systems. **Data
engineering** is the discipline of **collecting** this data, **cleaning** it,
**organizing** it, and **delivering** it to the people and tools that need it —
reliably and on time.

> **Analogy — the city's water system.** Data engineers are the plumbers of
> data. Rain (raw data) falls everywhere; engineers build reservoirs to store
> it, treatment plants to clean it, and pipes to carry clean water to every home
> (dashboards, reports, ML models). You rarely think about the pipes — but
> nothing works without them.

### The two worlds of data (the key split)

- **Operational systems (OLTP)** — *run* the day-to-day business: "Place the
  order. Charge the card. Update the stock."
- **Analytical systems (OLAP)** — *understand* the business: "Which product sold
  best last quarter? Which stores are shrinking?"

Almost every concept in this guide exists to move data from the **OLTP** world to
the **OLAP** world.

### Three flavours of data

| Type | What it means | Example | Fits neatly in a table? |
|---|---|---|---|
| **Structured** | Rows & columns, fixed shape | A sales table; a bank statement | Yes — made for it |
| **Semi-structured** | Some structure via tags/keys, flexible | JSON, XML, CSV, logs | Partly — needs parsing |
| **Unstructured** | No predefined structure | Images, audio, video, free-text reviews | No |

A warehouse loves *structured* data; a data lake was invented to also hold
*semi-* and *unstructured* data. Keep this in mind — it's the "why" behind
warehouses, lakes, and lakehouses.

## 2 · OLTP — Online Transaction Processing  ·  *runs the business*

OLTP systems handle the live, everyday **transactions** — the small, fast
operations that happen constantly. When you swipe your card or add an item to a
cart, an OLTP system is doing the work.

> **Analogy — the cashier at the counter.** It handles one customer at a time,
> very quickly: scan, pay, receipt, next please. It is never asked "what were our
> total sales last year?" — that would hold up the queue.

**Good at:** fast reads/writes of single records (milliseconds); many concurrent
users; **data integrity** via **ACID** (Atomicity, Consistency, Isolation,
Durability); storing what is true *right now*.

**Example — a slice of BrewBox's live `orders` table.** Note it is *normalized*:
a customer is stored by ID, not by full details, to avoid repetition. A typical
operation touches **one row** ("insert order 100873", "update its status to
PAID").

In [ ]:
oltp_orders = spark.createDataFrame([
    (100871,"C-4021","S-12","Latte",     1, 4.50,"PAID"),
    (100872,"C-3888","S-07","Cappuccino",2, 9.00,"PAID"),
    (100873,"C-4021","S-12","Muffin",    1, 3.25,"PENDING"),
    (100874,"C-5567","S-31","Cold Brew", 1, 5.00,"PAID"),
], ["order_id","customer_id","store_id","item","qty","amount","status"])

oltp_orders.show()
# Common OLTP databases: PostgreSQL, MySQL, Oracle, SQL Server (and NoSQL like MongoDB).

## 3 · OLAP — Online Analytical Processing  ·  *understands the business*

OLAP systems are built for **analysis**: looking across huge amounts of
historical data to find patterns, trends, and totals. Where OLTP *writes small
and often*, OLAP *reads big and periodically*.

> **Analogy — the store manager's monthly review.** She sits with every receipt
> and asks big questions: "Which drink is the bestseller? Which region is
> growing?" That reflective, aggregate view is OLAP.

**Good at:** aggregations (sums/averages/counts over millions of rows); years of
history; slicing & dicing by time/product/region; read-heavy workloads.

**Example — OLAP is often pre-summarized.** Behind each number below sit tens of
thousands of the tiny OLTP rows above — OLAP *collected, grouped, and summarized*
them. This is exactly what a `GROUP BY` produces.

In [ ]:
olap_summary = spark.createDataFrame([
    ("2026-06","North","Latte",    48230,217035.0,4.50),
    ("2026-06","North","Cold Brew",19540, 97700.0,5.00),
    ("2026-06","South","Latte",    52110,234495.0,4.50),
    ("2026-06","South","Cappuccino",33900,152550.0,4.50),
], ["month","region","product","units_sold","revenue","avg_order_value"])

olap_summary.show()
# Analysts picture OLAP data as a "cube": slice (just June), dice (June+North+Latte),
# drill down (year->month->day), roll up (store->region->country).
# Common OLAP systems: Snowflake, BigQuery, Redshift, Synapse, Databricks SQL.

## 4 · OLTP vs OLAP — side by side

This single comparison is worth memorizing. Almost every later concept exists to
move data from the OLTP world into the OLAP world.

| Aspect | OLTP (Transactional) | OLAP (Analytical) |
|---|---|---|
| **Purpose** | Run daily operations | Analyze & report |
| **Typical question** | "Charge this order." | "What were Q2 sales by region?" |
| **Operations** | Many small INSERT/UPDATE/DELETE | Large SELECT with GROUP BY |
| **Rows per query** | One or a few | Thousands to millions |
| **Data age** | Current state | Years of history |
| **Users** | Customers, apps (thousands live) | Analysts, managers (fewer) |
| **Storage layout** | **Row-based** | **Column-based** |
| **Design** | Normalized (no repetition) | Denormalized (built for reading) |
| **Examples** | MySQL, PostgreSQL, Oracle | Snowflake, BigQuery, Redshift |

### Row storage vs column storage (a favourite interview topic)

Take four order rows with columns `id, product, amount`:

- **Row storage (OLTP)** keeps each row together on disk → fetching one *whole
  order* is fast.
- **Column storage (OLAP)** keeps each *column* together → summing *every amount*
  reads just that one column and skips the rest.

```
Row storage (OLTP):     [1,Latte,4.50] [2,Muffin,3.25] [3,Latte,4.50] ...
Column storage (OLAP):  ids:[1,2,3,4]  products:[Latte,Muffin,Latte,...]  amounts:[4.50,3.25,...]
```

That's why analytical engines (and Parquet/Delta) store data **by column** —
massive aggregations become dramatically faster.

> **Takeaway:** OLTP and OLAP are *teammates*, not competitors. The whole data
> platform exists to move and reshape data from the OLTP side to the OLAP side.

## 5 · Data Warehouse — the organized library

A **data warehouse** is a large, central database built specifically for
**analytics (OLAP)**. It gathers data from many operational systems, cleans and
structures it, and stores it in a consistent, query-friendly format.

> **Analogy — a well-run library.** Books arrive from many publishers in
> different shapes and languages. Librarians clean them, translate them into one
> language, label them consistently, and shelve them by category. Now anyone can
> find what they need in minutes.

**Key trait — schema-on-write.** A warehouse decides the structure *before*
loading. Data must be cleaned and fit a defined table shape (a **schema**) on the
way in. Highly trustworthy and fast to query — but less flexible for messy data.

| Strengths | Limitations |
|---|---|
| Very fast analytical queries | Mostly structured (table) data |
| Clean, consistent, trustworthy | Rigid — schema defined up front |
| Great for BI dashboards | Storing huge raw volumes is costly |
| Strong governance & security | Not ideal for images/video/logs |

**Examples:** Snowflake, Amazon Redshift, Google BigQuery, Azure Synapse. *(How a
warehouse is organized internally — star & snowflake schemas — gets its own
section below.)*

## 6 · Data Lake — the giant reservoir

A **data lake** is a huge, low-cost storage repository that holds **any** kind of
data in its **raw** form — structured tables, semi-structured JSON/CSV, and
unstructured images/audio/video/logs. You pour data in first and decide how to
use it later.

> **Analogy — a giant natural lake.** Rivers and rain (all your sources) flow in
> freely, in whatever form they arrive. Nothing is filtered on the way in. When
> you need water, you take some out and treat it for its purpose.

**Key trait — schema-on-read** (the opposite of a warehouse): store data as-is,
apply structure only when you *read* it. Flexible and cheap — but without
discipline a lake becomes a **"data swamp"** nobody trusts.

**Example — a lake is really a big file store** (folders of files in cloud
storage), holding many data types side by side:

| Path / object | Type | Example use |
|---|---|---|
| `/raw/orders/2026-07-22.json` | Semi-structured | Daily order exports |
| `/raw/clickstream/*.log` | Unstructured logs | Website behavior analysis |
| `/raw/reviews/*.csv` | Structured text | Customer sentiment |
| `/raw/store_photos/*.jpg` | Unstructured images | Computer-vision shelf checks |

| Strengths | Limitations |
|---|---|
| Stores any data type | Easy to become a messy "swamp" |
| Very cheap storage at scale | Weak data quality/governance by default |
| Flexible — no schema up front | Slower/harder to query than a warehouse |
| Great for ML & data science | No built-in transactions/reliability |

**Built on cloud object storage:** Amazon S3, Azure Data Lake Storage (ADLS),
Google Cloud Storage (GCS).

## 7 · Lakehouse — best of both worlds

For years companies ran **both** a lake (cheap, flexible, for raw data & ML) and
a warehouse (clean, fast, for BI) — two systems, two copies, many pipelines. The
**lakehouse** merges them: it adds the **reliability and structure of a
warehouse** directly on top of the **cheap, flexible storage of a lake**.

> **Analogy — a reservoir with a modern treatment plant built in.** One big cheap
> reservoir (the lake), with a high-quality treatment/bottling plant on top so
> you can also draw clean, reliable water on demand (the warehouse) — from one
> place.

**The secret ingredient — the open table format.** Technologies like **Delta
Lake**, **Apache Iceberg**, and **Apache Hudi** add a *transaction layer* over
plain files. This gives lake files warehouse-like superpowers: **ACID
transactions, schema enforcement, updates/deletes, and time travel** (querying
data as it looked last week).

| Feature | Warehouse | Lake | **Lakehouse** |
|---|---|---|---|
| Handles raw/unstructured data | No | Yes | **Yes** |
| Cheap storage | No | Yes | **Yes** |
| ACID transactions & reliability | Yes | No | **Yes** |
| Fast BI queries | Yes | Limited | **Yes** |
| Good for ML | Limited | Yes | **Yes** |
| Systems to run | Two (with a lake) | Two (with a warehouse) | **One** |

**Examples:** Databricks Lakehouse (Delta Lake), Snowflake with Iceberg, Iceberg
on any cloud. The lakehouse is the natural home for the **Medallion Architecture**
(Section 12).

**Example — a Delta table with time travel** (Databricks tables are Delta by
default):

In [ ]:
# Write a Delta table, change it, then look at its version history ("time travel").
spark.sql("CREATE SCHEMA IF NOT EXISTS de_intro")
olap_summary.write.format("delta").mode("overwrite").saveAsTable("de_intro.sales_summary")

spark.sql("UPDATE de_intro.sales_summary SET revenue = revenue * 1.1 WHERE region = 'North'")
print("Version history (every change is a snapshot you can roll back to):")
spark.sql("DESCRIBE HISTORY de_intro.sales_summary").select("version","operation").show(truncate=False)

# Read the ORIGINAL version — this is time travel:
spark.read.option("versionAsOf", 0).table("de_intro.sales_summary").show()

## 8 · Data modeling — fact & dimension tables

Inside a warehouse/lakehouse, how do you *organize* analytical tables? With
**dimensional modeling**. It splits data into two kinds of table:

### Fact tables — the numbers you *measure*
A **fact table** stores **measurable business events**, one row per event at a
chosen **grain** (level of detail — e.g., one row per sale line). Its columns are:

- **Measures** — the numbers you aggregate: `units`, `revenue`, `discount`.
- **Foreign keys** — links to dimensions: `product_key`, `store_key`, `date_key`.

Fact tables are long and thin (billions of rows, few columns) and are usually
**additive** (you can `SUM` the measures).

### Dimension tables — the descriptive *context*
A **dimension table** stores the **who / what / where / when** that gives facts
meaning: `dim_product`, `dim_store`, `dim_date`, `dim_customer`. Each has a
**key** and descriptive **attributes** (name, category, region…). Dimensions are
short and wide (few rows, many descriptive columns).

> **The point:** by storing `"Hot Coffee"` once in `dim_product` and referencing
> it by `product_key` in a billion fact rows, you save space *and* can ask
> "revenue by category" with a simple join. Facts = **verbs/numbers**, dimensions
> = **nouns/adjectives**.

**Example — build a tiny fact + dimensions and join them:**

In [ ]:
fact_sales = spark.createDataFrame([
    (9001,"2026-06-22","P-01","S-12",3,13.50),
    (9002,"2026-06-22","P-04","S-07",1, 5.00),
    (9003,"2026-06-23","P-01","S-31",2, 9.00),
    (9004,"2026-06-23","P-04","S-12",4,20.00),
], ["sale_id","date_key","product_key","store_key","units","revenue"])

dim_product = spark.createDataFrame([
    ("P-01","Latte","Hot Coffee","Medium"),
    ("P-04","Cold Brew","Cold Coffee","Large"),
], ["product_key","name","category","size"])

dim_store = spark.createDataFrame([
    ("S-07","Downtown","North"),
    ("S-12","Airport","North"),
    ("S-31","Harbour","South"),
], ["store_key","store_name","region"])

# Join the FACT to its DIMENSIONS, then aggregate a measure by dimension attributes
(fact_sales
    .join(dim_product, "product_key")
    .join(dim_store,   "store_key")
    .groupBy("category","region")
    .agg(F.sum("revenue").alias("revenue"))
    .orderBy("category","region")
    .show())

## 9 · Star schema

A **star schema** puts one central **fact** table in the middle, surrounded by
**dimension** tables — the diagram looks like a star. Crucially, the dimensions
are **denormalized**: each dimension is a single flat table (e.g., `dim_product`
holds `category` and `size` right in it, no further tables).

```
                 dim_date
                    |
   dim_store --- fact_sales --- dim_product
                    |
                dim_customer
```

- ✅ **Simple** — few joins, easy for analysts and BI tools to understand.
- ✅ **Fast** — fewer joins means quick queries (the usual choice for BI).
- ⚠️ **Some redundancy** — e.g., the category name repeats for every product in
  that category.

**Example:** the join in Section 8 *is* a star-schema query — `fact_sales`
joined directly to flat `dim_product` and `dim_store`. That's the star: fact in
the middle, one hop to each dimension.

## 10 · Snowflake schema

A **snowflake schema** is a star schema whose dimensions are **normalized** —
split into sub-dimension tables to remove redundancy. `dim_product` no longer
stores the category *text*; it stores a `category_key` that points to a separate
`dim_category` table. The diagram branches out like a snowflake.

```
   dim_category
        |
   dim_product --- fact_sales --- dim_store --- dim_region
                       |
                    dim_date
```

**Star vs snowflake — the trade-off:**

| | **Star** | **Snowflake** |
|---|---|---|
| Dimensions | Denormalized (flat) | Normalized (split into sub-tables) |
| Joins per query | Fewer | More |
| Redundancy | Some | Minimal |
| Query speed | Faster | Slightly slower (more joins) |
| Storage | Slightly more | Slightly less |
| Best for | BI/dashboards (most common) | Large, complex dimensions; strict consistency |

**Example — normalize `dim_product` into `dim_product` + `dim_category`, then it
takes an extra join to get the category:**

In [ ]:
# Snowflaked product dimension: category text lives in its OWN table
dim_product_snow = spark.createDataFrame([
    ("P-01","Latte","C-1","Medium"),
    ("P-04","Cold Brew","C-2","Large"),
], ["product_key","name","category_key","size"])

dim_category = spark.createDataFrame([
    ("C-1","Hot Coffee","Beverages"),
    ("C-2","Cold Coffee","Beverages"),
], ["category_key","category_name","department"])

# Now the fact must hop fact -> dim_product_snow -> dim_category (one extra join)
(fact_sales
    .join(dim_product_snow, "product_key")
    .join(dim_category,     "category_key")
    .groupBy("department","category_name")
    .agg(F.sum("revenue").alias("revenue"))
    .orderBy("category_name")
    .show())

## 11 · ETL — Extract, Transform, Load

How does data get from sources (OLTP, apps, files) to destinations (warehouse/
lake), cleaned along the way? The classic answer is **ETL**:

1. **Extract** — pull raw data out of source systems.
2. **Transform** — clean, reshape, combine, and validate it *before* it lands.
3. **Load** — write the finished, clean data into the destination.

> **Analogy — a restaurant kitchen.** Extract = collect raw ingredients;
> Transform = wash, chop, cook, plate; Load = serve the finished dish. The diner
> only ever sees the clean result — never the muddy vegetables.

**The defining trait of ETL: data is transformed *before* it is loaded.** The
destination only ever holds clean data.

**Example — raw BrewBox data is messy** (inconsistent casing, a missing price, a
hard-to-read epoch timestamp, varied city spelling). The Transform step fixes all
of it before loading:

In [ ]:
raw = spark.createDataFrame([
    (1,"latte",     4.5, "london", 1657872842),
    (2,"LATTE",     4.5, "London", 1657872845),
    (3,"Cold Brew", None,"LONDON", 1657872849),
], ["id","product","amount","city","ts"])

price_list = spark.createDataFrame([("Latte",4.50),("Cold Brew",5.00)], ["product","list_price"])

# TRANSFORM: title-case product, fill missing price, standardize city, readable time
clean = (raw
    .withColumn("product", F.initcap("product"))
    .join(price_list, "product", "left")
    .withColumn("amount", F.coalesce("amount","list_price"))
    .withColumn("city", F.initcap("city"))
    .withColumn("sale_time", F.from_unixtime("ts").cast("timestamp"))
    .select("id","product","amount","city","sale_time"))

clean.show(truncate=False)   # ...only THIS clean result is loaded into the warehouse
# Classic ETL tools: Informatica, Talend, SSIS.

## 12 · ELT — Extract, Load, Transform

**ELT** is the modern reordering: extract raw data and **load it immediately**
into a powerful destination (cloud warehouse/lakehouse), then **transform it
there** using the destination's own massive compute.

> **Analogy — a meal-kit service.** Instead of a chef cooking everything first
> (ETL), the company ships all the raw ingredients to your kitchen (Extract →
> Load), and you cook exactly what you want, when you want (Transform, later). You
> keep the raw ingredients, so you can cook something different tomorrow.

**Why ELT won:** cloud storage became cheap and cloud compute became powerful, so
it makes sense to land raw data first and transform in place. You **keep the raw
data**, so you can re-transform it differently later (great for ML).

| Aspect | **ETL** | **ELT** |
|---|---|---|
| Order | Extract → **Transform** → Load | Extract → Load → **Transform** |
| Where transform happens | Separate tool, before loading | Inside the destination, after loading |
| Destination gets | Only clean data | Raw first, then cleaned in place |
| Raw data kept? | Usually discarded | **Yes** — stored and reusable |
| Best fit | On-prem, limited compute, pre-cleaning/masking | Cloud warehouses & lakehouses |
| Typical tools | Informatica, Talend, SSIS | **dbt**, Fivetran + Snowflake/BigQuery/Databricks |

**Example — ELT: load raw as-is, then transform *in place* with SQL** (the way
dbt or a Databricks job would):

In [ ]:
# EXTRACT + LOAD: land the raw data untouched as a table (no cleaning yet)
raw.write.format("delta").mode("overwrite").saveAsTable("de_intro.orders_raw")

# TRANSFORM in place, inside the destination, using its own compute (SQL here):
spark.sql("""
    CREATE OR REPLACE TABLE de_intro.orders_clean AS
    SELECT id,
           initcap(product)                         AS product,
           coalesce(amount, 5.00)                   AS amount,
           initcap(city)                            AS city,
           cast(from_unixtime(ts) AS timestamp)     AS sale_time
    FROM de_intro.orders_raw
""")
spark.table("de_intro.orders_clean").show(truncate=False)
# The raw table is still there to re-transform differently later.

## 13 · Apache Spark — the engine that crunches big data

When you have **billions of rows** — more than one machine's memory or CPU can
handle — you need many computers working together. **Apache Spark** is the most
popular open-source engine for processing huge datasets **in parallel** across a
**cluster**.

> **Analogy — counting votes.** One person counting 10 million ballots takes
> weeks. Split them across 1,000 volunteers who each count their pile at the same
> time, then combine subtotals. Spark is the organizer that splits the work,
> hands pieces to many machines, and merges the results.

**Key idea — distributed, in-memory processing.** Spark splits a big job into
small tasks, runs them on many machines at once, and keeps data in **memory
(RAM)** where possible. A cluster has a **driver** (plans the job) and many
**executors** (do the work); data is split into **partitions**, one per task.

| Worker (executor) | Rows | Local subtotal (revenue) |
|---|---|---|
| Executor 1 (orders A–F) | 250 M | \$1.12 B |
| Executor 2 (orders G–M) | 250 M | \$0.98 B |
| Executor 3 (orders N–S) | 250 M | \$1.05 B |
| Executor 4 (orders T–Z) | 250 M | \$1.01 B |
| **Driver combines →** | 1 B | **\$4.16 B total** |

**Used for:** batch processing (the "T" in big-data ETL/ELT), streaming (Spark
Structured Streaming), machine learning (MLlib), and SQL analytics (Spark SQL).
It's the engine under **Databricks** and Amazon EMR.

**Example — see how Spark partitions data across the cluster:**

In [ ]:
df = spark.range(0, 1_000_000)                       # a lazy, distributed dataset
print("partitions (chunks processed in parallel):", df.rdd.getNumPartitions())
print("count (an ACTION that triggers the distributed job):", df.count())
# Transformations (select/filter/groupBy) are LAZY - they build a plan.
# Actions (count/show/collect/write) trigger Spark to actually run it.

## 14 · Data pipeline — moving data from A to B

A **data pipeline** is the end-to-end series of **automated** steps that moves
data from sources, through processing, to its destination — reliably and on a
schedule, without a human doing it by hand. **ETL/ELT are pipeline patterns;
Spark is often the engine inside them.**

> **Analogy — a factory conveyor belt.** Raw materials go in one end; each station
> does one job (inspect, wash, assemble, paint, package); a finished product
> rolls off the other end — automatically, the same way every time.

**Stages of a typical pipeline (BrewBox's daily sales):**

| Stage | What happens | Example |
|---|---|---|
| 1. Ingest | Pull data from sources | Export yesterday's orders from OLTP & the app API |
| 2. Store (raw) | Land raw data cheaply | Drop JSON files into the data lake (**Bronze**) |
| 3. Process | Clean & transform (often Spark) | Standardize products, fix prices, join store info (**Silver**) |
| 4. Serve | Load into warehouse/lakehouse | Write clean/aggregated sales (**Gold**) |
| 5. Consume | Dashboards, reports, ML | Manager opens the daily sales dashboard |

**Batch vs streaming:**

| Aspect | **Batch** | **Streaming** |
|---|---|---|
| When it runs | On a schedule (e.g., nightly) | Continuously, as data arrives |
| Data handled | Big chunks at once | Small events one by one |
| Freshness | Hours old | Seconds old |
| Example | Nightly sales report | Live fraud detection |

**Orchestration.** Pipelines have many steps that must run in the right order, on
schedule, with retries on failure. **Orchestrators** — **Apache Airflow**,
**Dagster**, **Prefect** (and Databricks Workflows) — are the conductor: they
start each step at the right time and alert engineers when something breaks.

## 15 · Medallion Architecture — Bronze → Silver → Gold

Inside a lakehouse, how do you keep raw, clean, and business-ready data
organized? The **Medallion Architecture** (championed by Databricks) organizes
data into three progressively cleaner layers named after medals. Data flows
through them, getting more refined and more valuable at each step.

> **Analogy — refining gold ore.** **Bronze** is raw ore straight from the mine
> (dirty but complete). **Silver** is washed ore with impurities removed (clean,
> usable). **Gold** is the polished jewelry ready for the shop window.

| Layer | State of data | Who uses it | Purpose |
|---|---|---|---|
| 🥉 **Bronze** | Raw, unfiltered | Data engineers | Complete history; safe to reprocess |
| 🥈 **Silver** | Clean, validated, joined | Engineers, data scientists | Trusted single source of truth |
| 🥇 **Gold** | Aggregated, business-shaped | Analysts, managers, BI, ML | Fast, ready-to-use insights |

**Why teams love it:** clarity (everyone knows what quality to expect), safety
(rebuild Silver/Gold from untouched Bronze if a bug appears), reusability (one
Silver feeds many Gold tables), and incremental quality (fix problems step by
step).

**Example — a full Bronze → Silver → Gold pipeline** on the messy orders:

In [ ]:
# 🥉 BRONZE — land raw data exactly as it arrived (note: a duplicate + a null + epoch ts)
bronze = spark.createDataFrame([
    (1,"latte",     4.5, "london", 1657872842),
    (2,"latte",     4.5, "london", 1657872842),   # duplicate
    (3,"Cold Brew", None,"LONDON", 1657872849),
], ["id","product","amount","city","ts"])
bronze.write.format("delta").mode("overwrite").saveAsTable("de_intro.orders_bronze")
print("BRONZE (raw, warts and all):"); bronze.show()

In [ ]:
# 🥈 SILVER — clean, dedupe, fix, enrich (join a region lookup) -> single source of truth
region_lookup = spark.createDataFrame([("London","North"),("Manchester","South")], ["city","region"])

silver = (spark.table("de_intro.orders_bronze")
    .dropDuplicates(["product","amount","city","ts"])       # remove the duplicate
    .withColumn("product", F.initcap("product"))
    .withColumn("amount", F.coalesce("amount", F.lit(5.00))) # fill missing price
    .withColumn("city", F.initcap("city"))
    .withColumn("sale_time", F.from_unixtime("ts").cast("timestamp"))
    .join(region_lookup, "city", "left")                    # enrich with region
    .select("id","product","amount","city","region","sale_time"))

silver.write.format("delta").mode("overwrite").saveAsTable("de_intro.orders_silver")
print("SILVER (clean, deduped, enriched):"); silver.show(truncate=False)

In [ ]:
# 🥇 GOLD — aggregate into a business-ready table for the dashboard
gold = (spark.table("de_intro.orders_silver")
    .groupBy(F.to_date("sale_time").alias("date"), "region")
    .agg(F.count("*").alias("total_orders"),
         F.round(F.sum("amount"), 2).alias("total_revenue")))

gold.write.format("delta").mode("overwrite").saveAsTable("de_intro.daily_revenue_gold")
print("GOLD (ready for BI):"); gold.show()

## 16 · How it all fits together

Let's trace **one cup of coffee's data journey** through everything above:

| Step | What happens | Concept in action |
|---|---|---|
| 1 | A customer buys a latte; the order is saved instantly | **OLTP** database (PostgreSQL) |
| 2 | Overnight, a job copies all orders out | **Data pipeline** — ingest/extract |
| 3 | Raw orders land untouched in cheap storage | **Data lake → Bronze** |
| 4 | A cluster cleans, dedupes, enriches the data | **Apache Spark → Silver** |
| 5 | Clean data is loaded, then aggregated in place | **ELT in a Lakehouse → Gold** |
| 6 | A manager opens the daily revenue dashboard | **OLAP** analytics on the Gold table |

> **The one-sentence summary of data engineering:** take raw data from the
> systems that run the business (**OLTP**), move it reliably through **pipelines**,
> store and refine it (**lake → lakehouse**, **Bronze → Silver → Gold**) using
> engines like **Spark**, and deliver clean, ready answers to the systems that
> help you understand the business (**OLAP**).

## 17 · The technology landscape — what each tool is *for*

Job posts throw many tool names at you. Here's how the popular ones map onto the
concepts above — master the **concept**, learn **one tool per row**, and the rest
come quickly because they do the same underlying job.

| Job in the pipeline | Concept | Popular tools |
|---|---|---|
| Run the app / capture data | **OLTP** | PostgreSQL, MySQL, Oracle, MongoDB |
| Cheap raw storage | **Data Lake** | Amazon S3, Azure ADLS, Google GCS |
| Analytics warehouse | **Data Warehouse / OLAP** | Snowflake, BigQuery, Redshift, Synapse |
| Unified lake + warehouse | **Lakehouse** | Databricks, Delta Lake, Apache Iceberg |
| Move data in (ingest) | **Extract / Load** | Fivetran, Airbyte, Kafka (streaming) |
| Transform data | **Transform (ELT/ETL)** | **dbt**, Apache Spark, SQL |
| Heavy big-data processing | **Distributed compute** | **Apache Spark** (Databricks / EMR) |
| Schedule & coordinate | **Orchestration** | Airflow, Dagster, Prefect, Databricks Workflows |
| Show the results | **BI / consumption** | Power BI, Tableau, Looker |

### The languages & libraries you'll actually write — and *why*

| Tool | What it is | When / why you use it |
|---|---|---|
| **SQL** | The query language of data | **Non-negotiable.** Every warehouse, lakehouse, dbt model and Spark job speaks it. Filtering, joining, aggregating, modeling — most transformation work is SQL. |
| **Python** | General-purpose language | The glue of data engineering: scripting pipelines, calling APIs, automation, orchestration (Airflow DAGs), and driving Spark via **PySpark**. |
| **pandas** | In-memory DataFrame library for Python | Great for **small-to-medium** data that fits on **one machine**: quick exploration, cleaning, prototyping, tests, and lightweight transforms. Simple and fast — until the data outgrows one computer. |
| **NumPy** | Numerical arrays under pandas | Vectorized math; the engine beneath pandas. You rarely use it directly, but it's why pandas is fast. |
| **PySpark** | Python API for Apache Spark | The **big-data** version of pandas: the *same* transformations (select/filter/join/groupBy) but executed **distributed** across a cluster. Reach for it when data is too large for one machine, or you're on Databricks. |
| **Spark SQL** | SQL on Spark | Run SQL over huge distributed tables; mixes freely with the DataFrame API — same engine underneath. |
| **dbt** | Transformation framework (the "T" of ELT) | Turns SQL `SELECT`s into tested, documented, dependency-ordered tables **inside** the warehouse/lakehouse. Brings software engineering (version control, tests, docs, CI) to analytics. The standard for ELT modeling. |
| **Delta Lake** | Open table format | Adds ACID transactions, updates/deletes/**MERGE**, and **time travel** to lake files — the foundation of the lakehouse and the Medallion layers. |
| **Airflow / Dagster / Prefect** | Orchestrators | Schedule multi-step pipelines, enforce run order, retry on failure, alert on breakage. The "conductor" of the whole pipeline. |
| **Kafka** | Streaming platform | Move events continuously in real time (the ingest layer for streaming pipelines). |

**Rule of thumb — pandas vs PySpark:** prototype and handle small/medium data
with **pandas**; switch to **PySpark / Spark** when it no longer fits on one
machine. The concepts transfer directly — that's why this course teaches both.

> How they work together on a typical day: **Fivetran/Kafka** ingest raw data →
> it lands as **Delta** (Bronze) on **S3/ADLS** → **Spark/PySpark** and **dbt**
> transform it (Silver → Gold) → **Airflow/Databricks Workflows** schedules it all
> → **Power BI/Tableau** and ML models consume the Gold tables. **SQL and Python**
> are the languages you write throughout.

## 18 · Roles, skills & a learning path

Data engineering sits alongside related roles — think of it as a **relay**: the
data engineer lays the track and delivers clean data; analysts, data scientists,
and ML engineers run with it.

| Role | Main focus | Typical output |
|---|---|---|
| **Data Engineer** | Build & run pipelines; move and shape data | Reliable, clean data tables |
| **Analytics Engineer** | Transform data (ELT) for analysts | Clean, modelled warehouse tables (dbt) |
| **Data Analyst** | Explore data; answer business questions | Dashboards, reports, insights |
| **Data Scientist** | Build statistical / ML models | Predictions, models, experiments |
| **ML Engineer** | Put models into production | Deployed, monitored ML systems |

**A practical learning path for a fresher:**

1. **SQL** — the universal language of data (used by every tool here).
2. **Python** — scripting, automation, and PySpark.
3. **A cloud platform** (AWS / Azure / GCP) — modern data lives in the cloud.
4. **Data modelling** — star/snowflake schemas, the Medallion layers.
5. **Apache Spark (PySpark)** — processing data too big for one machine.
6. **Orchestration (Airflow / Databricks Workflows)** — reliably running pipelines.

> **A first project to try:** take any public CSV (weather or sales). Land it raw
> in a folder (**Bronze**), clean it into a tidy table (**Silver**), then produce
> a summary like "average by month" (**Gold**). You've just built a Medallion
> pipeline and touched most concepts in this guide.

**Interview tip:** if you can confidently explain the **OLTP → OLAP** journey and
where each concept (pipeline, lake, Spark, ELT, Medallion) fits — using a simple
example like the coffee shop — you'll handle most fresher-level DE interviews.

## 19 · Glossary

| Term | Plain-English meaning |
|---|---|
| **OLTP** | Online Transaction Processing — fast, live, everyday transactions that run the business. |
| **OLAP** | Online Analytical Processing — analyzing large amounts of historical data for insights. |
| **ACID** | Atomicity, Consistency, Isolation, Durability — guarantees that keep transactions correct. |
| **Data Warehouse** | Central structured database for analytics; cleaned before loading (schema-on-write). |
| **Data Lake** | Cheap storage for any raw data; structure applied when read (schema-on-read). |
| **Lakehouse** | A lake's flexibility/cost + a warehouse's reliability/structure, in one system. |
| **Schema** | The defined structure of data — its columns, types, and rules. |
| **Fact table** | The numbers you measure (sales/units/revenue), one row per event at a grain. |
| **Dimension table** | Descriptive context (product, store, date) referenced by facts. |
| **Star schema** | Central fact + flat (denormalized) dimensions. |
| **Snowflake schema** | Star schema with normalized (split) dimensions. |
| **ETL** | Extract, Transform, Load — clean data *before* loading. |
| **ELT** | Extract, Load, Transform — load raw first, then transform in the destination. |
| **Apache Spark** | Engine that processes huge datasets in parallel across a cluster. |
| **Cluster / Partition** | A group of machines working as one / a chunk of data one worker processes. |
| **Data Pipeline** | Automated, scheduled steps moving data from source to destination. |
| **Batch / Streaming** | Scheduled chunks / continuous per-event processing. |
| **Orchestrator** | Tool (Airflow, Dagster) that schedules and coordinates pipeline steps. |
| **Medallion Architecture** | Bronze (raw) → Silver (clean) → Gold (business-ready) layers. |
| **Time Travel** | Querying data as it looked at an earlier point in time (a lakehouse feature). |
| **BI** | Business Intelligence — dashboards and reports that turn data into decisions. |

## 🎉 Congratulations

You now understand the core vocabulary and architecture of modern data
engineering — from the tiny transaction at a coffee counter all the way to the
executive dashboard. Keep this notebook handy as a reference, run the examples,
and start building.